In [1]:
import os
import nibabel as nib
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact, IntSlider

ADNI_BASE_PATH = os.getenv("ADNI_BASE_PATH")
ADNI_FLAT = f'{ADNI_BASE_PATH}/ADNI_flat'
ADNI_CNN = f'{ADNI_BASE_PATH}/ADNI_CNN_tensors'
ADNI_FLAT

'/Volumes/Corsair/agpmd/ADNI_flat'

In [9]:
# subject_id = "012_S_4026"
# subject_id = "067_S_6045"
# subject_id = "016_S_4688"
subject_id = "032_S_4277"

In [13]:
mri_path = f'{ADNI_FLAT}/{subject_id}/mri.nii'
pet_path = f'{ADNI_FLAT}/{subject_id}/pet.nii.gz'

print("Loading NIfTI files...")
mri_img = nib.load(mri_path)
pet_img = nib.load(pet_path)

mri_data = mri_img.get_fdata()
pet_data = pet_img.get_fdata()

if mri_data.ndim > 3: mri_data = np.squeeze(mri_data)
if pet_data.ndim > 3: pet_data = np.squeeze(pet_data)

print(f"MRI Shape: {mri_data.shape}")
print(f"PET Shape: {pet_data.shape}")

def explore_3d_images(mri_z, pet_z):
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    axes[0].imshow(mri_data[:, :, mri_z].T, cmap='gray', origin='lower')
    axes[0].set_title(f'MRI (MT1 N3m) - Axial Slice {mri_z}')
    axes[0].axis('off')

    axes[1].imshow(pet_data[:, :, pet_z].T, cmap='hot', origin='lower')
    axes[1].set_title(f'PET (AV45 Amyloid) - Axial Slice {pet_z}')
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()

mri_max = mri_data.shape[2] - 1
pet_max = pet_data.shape[2] - 1

print("\nUse the sliders below to scroll through the brain volumes:")
interact(
    explore_3d_images,
    mri_z=IntSlider(min=0, max=mri_max, step=1, value=mri_max//2, description='MRI Slice'),
    pet_z=IntSlider(min=0, max=pet_max, step=1, value=pet_max//2, description='PET Slice')
);

Loading NIfTI files...
MRI Shape: (176, 240, 256)
PET Shape: (160, 160, 96)

Use the sliders below to scroll through the brain volumes:


interactive(children=(IntSlider(value=127, description='MRI Slice', max=255), IntSlider(value=47, description=…

In [14]:
harmonized_mri_path = f'{ADNI_CNN}/{subject_id}_mri.npy'
harmonized_pet_path = f'{ADNI_CNN}/{subject_id}_pet.npy'

print("Loading CNN-Ready Tensors...")
# Using mmap_mode='r' simulates exactly how your PyTorch DataLoader will read them
harmonized_mri_data = np.load(harmonized_mri_path, mmap_mode='r')
harmonized_pet_data = np.load(harmonized_pet_path, mmap_mode='r')

print(f"Harmonized MRI Shape: {harmonized_mri_data.shape} | Min: {harmonized_mri_data.min():.2f}, Max: {harmonized_mri_data.max():.2f}")
print(f"Harmonized PET Shape: {harmonized_pet_data.shape} | Min: {harmonized_pet_data.min():.2f}, Max: {harmonized_pet_data.max():.2f}")

def inspect_harmonized_tensors(harmonized_z_slice):
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    axes[0].imshow(harmonized_mri_data[:, :, harmonized_z_slice].T, cmap='gray', origin='lower')
    axes[0].set_title(f'Warped MRI - Slice {harmonized_z_slice}')
    axes[0].axis('off')

    axes[1].imshow(harmonized_pet_data[:, :, harmonized_z_slice].T, cmap='hot', origin='lower')
    axes[1].set_title(f'Warped PET - Slice {harmonized_z_slice}')
    axes[1].axis('off')

    axes[2].imshow(harmonized_mri_data[:, :, harmonized_z_slice].T, cmap='gray', origin='lower')
    axes[2].imshow(harmonized_pet_data[:, :, harmonized_z_slice].T, cmap='hot', origin='lower', alpha=0.4,
                   vmin=0, vmax=1)
    axes[2].set_title('Co-Registration Overlay')
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

harmonized_max_depth = harmonized_mri_data.shape[2] - 1

print("\nUse the shared slider to explore the synchronized spatial grid:")
interact(
    inspect_harmonized_tensors,
    harmonized_z_slice=IntSlider(min=0, max=harmonized_max_depth, step=1, value=harmonized_max_depth//2, description='Shared Z-Axis')
);

Loading CNN-Ready Tensors...
Harmonized MRI Shape: (128, 128, 128) | Min: 0.00, Max: 1.00
Harmonized PET Shape: (128, 128, 128) | Min: 0.00, Max: 1.00

Use the shared slider to explore the synchronized spatial grid:


interactive(children=(IntSlider(value=63, description='Shared Z-Axis', max=127), Output()), _dom_classes=('wid…